# **Cleaning and Analysing Retail Sales Data Using PostgreSQL**

![project banner](images/banner_image.png)

## **Project Overview**

This project focuses on cleaning and analysing retail sales data using PostgreSQL. The dataset represents transactional records from a Super Store business and includes information about customer orders, products, sales performance, discounts, profits, and returns.

The primary objective of the project is to apply SQL techniques commonly used by data scientists and data analysts to solve real-world business problems involving data quality, aggregation, ranking, and missing value imputation.

Two major analytical tasks were completed during the project:

### Task 1 — Product Performance Analysis

The first task involved identifying the top five products within each product category based on total sales performance. SQL aggregation functions, JOIN operations, and window functions such as RANK() were used to calculate product rankings and profitability metrics.

This analysis helps businesses:
- Identify high-performing products
- Understand category-level sales trends
- Evaluate product profitability
- Support inventory and sales strategy decisions

### Task 2 — Missing Data Imputation

The second task focused on handling missing values in the quantity column within the orders dataset. Using existing order information such as sales values, discounts, market, and regional pricing patterns, unit prices were calculated and used to estimate missing quantities.

This task demonstrated practical data-cleaning techniques frequently required in:
- Data preprocessing
- Data quality improvement
- ETL pipelines
- Business intelligence workflows
- Machine learning preparation

Throughout the project, PostgreSQL was used to perform:
- Data cleaning
- Data transformation
- Aggregation and summarisation
- Window-based ranking
- Common Table Expressions (CTEs)
- Missing value estimation

The project provides hands-on experience with SQL-based data analysis and demonstrates how structured query language can be used to generate actionable business insights from raw retail datasets.

---

## **Data Description**

The dataset used in this project contains retail transaction data from a Super Store database. The data is distributed across multiple related tables that store information about orders, products, returns, and sales representatives.

### 1. Orders Table

The `orders` table contains transactional sales data for customer purchases.

| Column       | Description                           |
| ------------ | ------------------------------------- |
| `row_id`     | Unique identifier for each record     |
| `order_id`   | Unique identifier for each order      |
| `order_date` | Date when the order was placed        |
| `market`     | Market associated with the order      |
| `region`     | Geographic region of the customer     |
| `product_id` | Identifier of the purchased product   |
| `sales`      | Total sales amount for the order line |
| `quantity`   | Quantity of items purchased           |
| `discount`   | Discount applied to the order         |
| `profit`     | Profit generated from the order       |

---

### 2. Products Table

The `products` table stores product-level information and categorization details.

| Column         | Description                       |
| -------------- | --------------------------------- |
| `product_id`   | Unique identifier for the product |
| `category`     | Product category                  |
| `sub_category` | Product sub-category              |
| `product_name` | Detailed product name             |

---

### 3. Returned Orders Table

The `returned_orders` table contains information about returned products and orders.

| Column     | Description                               |
| ---------- | ----------------------------------------- |
| `returned` | Indicates whether the order was returned  |
| `order_id` | Identifier for the returned order         |
| `market`   | Market associated with the returned order |

---

### 4. People Table

The `people` table contains information about sales representatives and their operating regions.

| Column   | Description                           |
| -------- | ------------------------------------- |
| `person` | Name of the salesperson               |
| `region` | Region where the salesperson operates |

---

### Key Data Challenges

The dataset contains several realistic data quality challenges, including:

* Missing quantity values
* Text-based date fields
* Mixed numeric and text data types
* Multi-table relationships requiring joins
* Sales and profit aggregation requirements

---

In [2]:
import os
import pandas as pd

import requests
from io import StringIO
from io import BytesIO

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load env variables
load_dotenv()

host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
database = os.getenv("NEW_DATABASE")

In [3]:
# -----------------------------
# CONNECT TO DATABASE
# -----------------------------

db_url = (
    f"postgresql+psycopg://{user}:{password}@{host}:{port}/{database}"
)

engine = create_engine(db_url)

## **Task 1: Identify the top five products from each category with highest profit margin**
Find the top 5 products from each category based on highest total sales. The output should be sorted by category in ascending order and by sales in descending order within each category, i.e. within each category product with highest margin should sit on the top. Save the query as top_five_products_each_category, containing the following columns:
- category
- product_name
- product_total_sales (rounded to two decimal places)
- product_total_profit (rounded to two decimal places)
- product_rank 

### **Solution to Task 1**

Build a query to return the top 5 products in each of the categories.
- There are 3 categories.
- Top 5 products for each of the categories are calculated based on highest total sales.
- So, there should be 15 rows returned, five for each category.

Aggregating and ranking
- To start, you can SELECT * and use a subquery that selects the required columns from the orders and products tables. the query with Windows Function as a sub query and in the outer query use WHERE to only retain wanted number of rows
- Inside the subquery, select the category and product_name columns from the products table.
- Calculate product_total_profit and product_total_sales by taking the SUM() of the profit and sales columns from the orders table; CASTing to NUMERIC to successfully perform the calculation.
- You can wrap your calculations inside a call of ROUND() to round the results to 2 decimal places.
- To rank products by highest sales amounts, use the RANK() function with OVER(), PARTITIONing BY category and ORDERing BY the SUM() of sales in DESCending order.
- Perform an INNER JOIN to combine the orders and products tables, linking by product_id.
- Remember to use GROUP BY on the columns that you are not performing a calculation on.
- Remember to give your subquery a name, such as tmp, to avoid a syntax error.
- Outside of the subquery, use a WHERE keyword to to filter the product_rank column, only retaining values below 6 so you have the top five products per category.

In [4]:
task1_query = f"""
SELECT
    category,
    product_name,
    product_total_sales,
    product_total_profit,
    product_rank
FROM
(
    SELECT
        p.category,
        p.product_name,
	    ROUND(SUM(CAST(o.sales AS NUMERIC)), 2) AS product_total_sales,
	    ROUND(SUM(CAST(o.profit AS NUMERIC)), 2) AS product_total_profit,
	    RANK() OVER(PARTITION BY p.category ORDER BY SUM(CAST(o.sales AS NUMERIC)) DESC) AS product_rank
    FROM orders o
    INNER JOIN products p
        ON o.product_id = p.product_id
    GROUP BY
        p.category,
        p.product_name
) tmp
WHERE product_rank <= 5
ORDER BY
    category ASC,
    product_total_sales DESC;
"""

top_five_products_each_category = pd.read_sql(task1_query, engine)

print(top_five_products_each_category)

           category                                       product_name  \
0         Furniture         Hon Executive Leather Armchair, Adjustable   
1         Furniture  Office Star Executive Leather Armchair, Adjust...   
2         Furniture  Harbour Creations Executive Leather Armchair, ...   
3         Furniture            SAFCO Executive Leather Armchair, Black   
4         Furniture     Novimex Executive Leather Armchair, Adjustable   
5   Office Supplies                      Eldon File Cart, Single Width   
6   Office Supplies                                Hoover Stove, White   
7   Office Supplies                                  Hoover Stove, Red   
8   Office Supplies                     Rogers File Cart, Single Width   
9   Office Supplies                          Smead Lockers, Industrial   
10       Technology                       Apple Smart Phone, Full Size   
11       Technology                       Cisco Smart Phone, Full Size   
12       Technology                   

## **Task 2: Calculating missing quantity values**
Calculate quantity values where this field is missing.

Calculate the quantity for orders with missing values in the quantity column by determining the unit price for each product_id using available order data, considering relevant pricing factors such as discount, market, or region. Then, use this unit price to estimate the missing quantity values. The calculated values should be stored in the calculated_quantity column. Save query output as impute_missing_values, containing the following columns:
- product_id
- discount
- market
- region
- sales
- quantity
- calculated_quantity (rounded to zero decimal places)

### **Solution to Task 2**

General approach
- Find missing quantities: look for orders where the quantity is missing and store their details (product_id, discount, market, region, sales).
- Find other orders of the same product_id, discount, market, and region where quantity is available, then calculate the unit price (sales / quantity).
- Estimate missing quantities: use the unit price to estimate (impute) the missing quantity values by dividing sales by the unit price, rounding to the nearest whole number.

Structuring the query
- The query should have two CTEs: one to find all rows in orders where quantity values are missing, and another to calculate the unit price for each product.
- You can then calculate calculated_quantity and return all required columns for the output: product_id, discount, market, region, sales, quantity, calculated_quantity.

The first CTE: missing
- Create a CTE called missing to get all rows in the orders table with missing values for the quantity column.
- It should SELECT all columns required for the final output: product_id, discount, market, region, sales, quantity, calculated_quantity.
- Filter for rows where quantity is NULL.
- Once you have found the related information and calculate their unit price in 2nd CTE.
- In the main query join data from 1st CTE and 2nd CTEon product_id
  - Get all details from 1st CTE and calculate missing quantity by dividing sales from 1st CTE by unit_price from 2nd CTE

The second CTE: unit_prices
- In the second CTE, called unit_prices find product_id and alias a new column called unit_price, which will contain the price for a single quantity of each product.
- To perform the unit_price calculation you'll need to divide sales by quantity and wrap this in a CAST() function, converting to NUMERIC data type.
- The data should be SELECTed from the orders table, and then joined to the first CTE, missing.
- You need to join on product_id and discount to get related data only.

Finishing the query
- With both CTEs complete, you can SELECT all DISTINCT rows FROM the first CTE, missing.
- You then need add the calculated_quantity column to the output.
- To do this, you need to CAST() the sales column from the missing CTE as NUMERIC and divide this by the unit_price column from the unit_prices CTE.
- As you want calculated_quantity as whole numbers, you can ROUND() the results to 0 decimal places.
- Finish by using an INNER JOIN to the unit_prices CTE, joining on product_id.

In [5]:
task2_query = f"""
WITH missing AS (
	SELECT product_id,
		discount, 
		market,
		region,
		sales,
		quantity
	FROM orders 
	WHERE quantity IS NULL
), 

unit_prices AS (SELECT o.product_id,
	CAST(o.sales / o.quantity AS NUMERIC) AS unit_price
FROM orders o
RIGHT JOIN missing AS m 
	ON o.product_id = m.product_id
	AND o.discount = m.discount
WHERE o.quantity IS NOT NULL
)

SELECT DISTINCT m.*,
	ROUND(CAST(m.sales AS NUMERIC) / up.unit_price,0) AS calculated_quantity
FROM missing AS m
INNER JOIN unit_prices AS up
	ON m.product_id = up.product_id;
"""

impute_missing_values = pd.read_sql(task2_query, engine)

print(impute_missing_values)

         product_id  discount  market  region   sales quantity  \
0  FUR-ADV-10000571      0.00    EMEA    EMEA  438.96     None   
1  FUR-ADV-10004395      0.00    EMEA    EMEA   84.12     None   
2   FUR-BO-10001337      0.15      US    West  308.50     None   
3  TEC-STA-10003330      0.00  Africa  Africa  506.64     None   
4  TEC-STA-10004542      0.00  Africa  Africa  160.32     None   

   calculated_quantity  
0                  4.0  
1                  2.0  
2                  3.0  
3                  2.0  
4                  4.0  


## **Conclusion**
This project demonstrated how SQL can be effectively used for data cleaning, transformation, and business analysis within a retail sales database environment using PostgreSQL. The analysis focused on solving real-world data problems, including identifying top-performing products and handling missing quantity values through data imputation techniques.

In Task 1, SQL aggregation, JOIN operations, window functions, and ranking techniques were applied to identify the top five products in each category based on total sales performance. This provided valuable business insights into the products generating the highest revenue and profit across categories.

In Task 2, missing quantity values in the orders table were estimated using existing sales and pricing information. Common Table Expressions (CTEs), calculated unit prices, filtering, and conditional logic were used to derive estimated quantities for incomplete records. This demonstrated practical data-cleaning skills commonly required in data science and analytics workflows.

Overall, the project showcased important SQL concepts including:

Data cleaning and preprocessing
Handling missing values
Aggregation and grouping
JOIN operations
Window functions and ranking
Common Table Expressions (CTEs)
Numeric calculations and rounding
Creating reusable analytical tables

The project also highlighted the importance of data quality in analytics and how SQL can be used not only for querying databases but also for preparing reliable datasets for downstream business intelligence and machine learning applications.

By completing this project, strong foundational skills were developed in PostgreSQL-based data analysis, demonstrating the ability to transform raw retail data into meaningful business insights using structured query language (SQL).